# Gold — Cost by Environment

Custo mensal por ambiente com % do total.

**Métricas:**
- `total_cost_usd` — custo do ambiente no mês
- `pct_of_total` — percentual do custo total do mês

**Detecta desperdício** em ambientes não-produtivos (staging/dev com % alta = sinal de alerta).

In [ ]:
# parameters
execution_date = "2025-06-15"
year = 2025
month = 6
day = 15


In [ ]:
import sys
import os

notebooks_root = next(
    (p for p in ["/opt/airflow/notebooks", "/opt/spark/notebooks"]
     if __import__("os").path.exists(p)),
    "/opt/spark/notebooks",
)
if notebooks_root not in sys.path:
    sys.path.insert(0, notebooks_root)

from utils.spark_session import create_spark_session, postgres_jdbc_url, postgres_props
from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, DecimalType, TimestampType, DateType
)


In [ ]:
spark = create_spark_session("gold_cost_by_environment")

df_entries = spark.read.format("delta").load("s3a://datalake/silver/cost_entries/")

df_month_total = (
    df_entries
    .groupBy("year", "month")
    .agg(F.round(F.sum("cost_usd"), 4).alias("month_total_usd"))
)

df_by_env = (
    df_entries
    .groupBy("environment", "year", "month")
    .agg(F.round(F.sum("cost_usd"), 4).alias("total_cost_usd"))
)

df_gold = (
    df_by_env
    .join(df_month_total, on=["year", "month"], how="left")
    .withColumn(
        "pct_of_total",
        F.round((F.col("total_cost_usd") / F.col("month_total_usd")) * 100, 2),
    )
    .drop("month_total_usd")
    .withColumn("_processed_at", F.current_timestamp())
    .orderBy("year", "month", "environment")
)

output_path = "s3a://datalake/gold/cost_by_environment/"
df_gold.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(output_path)

print(f"[gold_cost_by_environment] {df_gold.count()} linhas → {output_path}")
df_gold.show(30)
spark.stop()
